#  **ICT303 - Assignment 2**

**Your name: Darynn Claire Ng**

**Student ID: 35302037**

**Email: darynnng2@gmail.com**

In [ ]:
# Install all required libraries before running any other cell
!pip install tensorflow numpy matplotlib seaborn scikit-learn pillow requests --quiet


All libraries installed successfully.


## 1. Problem Description

Skin diseases affect hundreds of millions of people worldwide. Accurate and timely diagnosis is critical for effective treatment outcomes. However, access to specialist dermatologists is limited — particularly in rural and low-income regions — and manual visual examination is subject to inter-observer variability and physician fatigue.

This project develops an **automated image classification system** capable of identifying **23 distinct skin disease categories** from dermoscopic photographs. The system is built on a custom Convolutional Neural Network (CNN) trained end-to-end directly on raw image pixels.

| Attribute | Detail |
|-----------|--------|
| **Task type** | Supervised multiclass image classification |
| **Input** | RGB dermoscopic photograph, resized to 128 × 128 pixels |
| **Output** | One of 23 disease class labels + softmax confidence score |
| **Model** | Custom 3-block CNN trained from scratch |

### Why CNNs for images?

Traditional machine learning models (SVM, random forests) require hand-crafted features — pixel histograms, texture descriptors, edge maps — which are brittle and domain-specific. CNNs eliminate this bottleneck by learning visual features **directly from raw pixels** through gradient descent.

A CNN processes an image in stages:
- **Early layers** detect primitive features: edges, colour gradients, simple textures
- **Middle layers** combine primitives into shapes: circles, blotches, lesion borders
- **Deep layers** recognise high-level patterns specific to each disease category

Crucially, convolutional filters **share weights** across spatial positions — the same edge detector applies at every location in the image. This gives CNNs *translation invariance* (a lesion in the corner is as recognisable as one in the centre) and makes them far more parameter-efficient than fully-connected networks applied directly to pixels.

## 2. Dataset Description

The dataset contains dermoscopic images spanning **23 disease classes**:

| # | Class | # | Class |
|---|-------|---|-------|
| 1 | Acne | 13 | Nail Fungus |
| 2 | Actinic Keratosis | 14 | Oil Seed |
| 3 | Atopic Dermatitis | 15 | Psoriasis |
| 4 | Bullous Disease | 16 | Scabies Lyme |
| 5 | Cellulitis Impetigo | 17 | Seborrheic Keratoses |
| 6 | Chicken Pox | 18 | Systemic Disease |
| 7 | Cowpox | 19 | Tinea Ringworm Candidiasis |
| 8 | HFMD | 20 | Urticaria Hives |
| 9 | Herpes HPV | 21 | Vascular Tumours |
| 10 | Light Diseases | 22 | Vasculitis |
| 11 | Lupus | 23 | Warts Molluscum |
| 12 | Melanoma | | |

### Dataset characteristics

- Images vary in resolution, lighting, camera angle, and skin tone across patients
- **Class imbalance** is present — some conditions (e.g. Melanoma) are better represented than others (e.g. Oil Seed, Bullous Disease)
- The dataset is divided into three non-overlapping splits:
  - **Training set (70%)** — used for weight optimisation
  - **Validation set (15%)** — monitored during training to tune hyperparameters and trigger callbacks
  - **Test set (15%)** — held out entirely; used only for final unbiased evaluation

### Preprocessing pipeline

All images pass through a fixed preprocessing pipeline before entering the network:

1. **Resize to 128 × 128 pixels** — establishes a uniform input dimension required by the CNN's fixed architecture. Larger inputs increase accuracy but multiply memory and compute cost quadratically.

2. **Pixel normalisation (÷ 255)** — rescales integer pixel values from [0, 255] to floating-point [0, 1]. This prevents large activation magnitudes in early layers, stabilises gradient flow, and accelerates convergence.

3. **Data augmentation (training only)** — during training, each image is randomly transformed before each epoch:
   - *Random horizontal flip* — simulates mirror-image presentations of the same lesion
   - *Random rotation ±15°* — accounts for variable camera orientation
   - *Random zoom ±10%* — simulates different distances between camera and skin

   Augmentation artificially multiplies the effective dataset size and teaches the model to be    invariant to these transformations, significantly reducing overfitting. It is **not applied    to validation or test sets** — those see only the normalised image.

## 2.5 Setting Up the Environment

Before building and training the model, all necessary tools and configuration must be prepared. This section covers four areas in separate cells: loading the external tools the project depends on, choosing the key numerical settings that control how the model learns, defining how images are randomly varied during training to improve generalisation, and loading the image dataset into efficient data streams.

In [ ]:
import os, numpy as np, warnings, sys
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

tf.random.set_seed(42)
np.random.seed(42)

print("All libraries imported successfully.")

### What the Imported Tools Do

This cell loads all the external tools the project relies on. The deep learning framework provides the building blocks for constructing, training, and saving the neural network, including the layer types, the optimiser, and the training loop. The numerical computation library handles all the large arrays of numbers that the model works with internally, such as pixel grids, weight matrices, and probability scores.

The visualisation library is used to draw the training progress graphs, the confusion matrix, the per-class bar chart, and the sample prediction gallery. The statistical visualisation library specifically provides the colour-coded heatmap used for the confusion matrix. The machine learning toolkit supplies ready-made functions for computing performance metrics such as precision, recall, and the full classification report.

Setting a fixed starting seed for both the deep learning framework and the numerical library ensures that results are reproducible across runs. Running the notebook twice with the same data will produce the same model and the same evaluation numbers.

In [ ]:
IMG_SIZE   = (128, 128)
BATCH_SIZE = 32
EPOCHS     = 25

CLASS_NAMES = [
    "Acne", "Actinic_Keratosis", "Atopic_Dermatitis", "Bullous_Disease",
    "Cellulitis_Impetigo", "Chicken_Pox", "Cowpox", "HFMD", "Herpes_HPV",
    "Light_Diseases", "Lupus", "Melanoma", "Nail_Fungus", "Oil_Seed",
    "Psoriasis", "Scabies_Lyme", "Seborrheic_Keratoses", "Systemic_Disease",
    "Tinea_Ringworm_Candidiasis", "Urticaria_Hives", "Vascular_Tumours",
    "Vasculitis", "Warts_Molluscum"
]
NUM_CLASSES = len(CLASS_NAMES)
print(f"Number of classes : {NUM_CLASSES}")
print(f"Image size        : {IMG_SIZE}")
print(f"Batch size        : {BATCH_SIZE}")
print(f"Max epochs        : {EPOCHS}")

### Core Training Settings Explained

Three numbers control the fundamental behaviour of the training process. The image size setting determines that every photograph entering the model is resized to a uniform square before any learning takes place. Smaller images process faster but may lose fine visual detail; the chosen size balances training speed against the amount of diagnostic information preserved in each image.

The batch size controls how many images are examined together in each individual learning step. Rather than updating the model's internal settings after every single image, a small group is gathered, the average error across the group is calculated, and the settings are adjusted once. Using a group rather than individual images makes each update more stable and less affected by the noise in any single photograph.

The maximum number of rounds sets an upper limit on how many times the model is allowed to pass through the entire training image collection. An automatic stopping mechanism described later will end training earlier if the model stops improving, preventing wasted computation and the risk of overfitting.

The list of twenty-three disease names serves as the label vocabulary. It maps each internal numerical category code back to a human-readable disease name used in all charts and reports throughout the notebook.

In [ ]:
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.10),
], name="augmentation")

normalise = layers.Rescaling(1.0 / 255, name="normalise")

print("Augmentation pipeline defined: RandomFlip, RandomRotation(0.15), RandomZoom(0.10)")
print("Normalisation layer defined: pixel values divided by 255")

### Image Preparation: Augmentation and Normalisation

The augmentation pipeline artificially creates modified copies of each training image on the fly. When an image enters the training process, it is randomly flipped horizontally to simulate a mirror image, slightly rotated to simulate a camera held at a small angle, and gently zoomed to simulate varying distances between the camera and the skin. None of these modifications change the underlying disease being depicted, but they expose the model to more variety during training. The practical benefit is that the model becomes less sensitive to exact framing, orientation, and scale, improving its performance on new photographs taken under different conditions.

These random modifications are applied only during training. When the model is being evaluated or used for predictions, every image passes through unchanged, so the evaluation numbers reflect performance on clean, unaltered images and give a fair picture of real-world capability.

The normalisation step divides all pixel brightness values by two hundred and fifty-five, the maximum possible value in a standard colour image. This brings all values into a narrow range between zero and one. Operating in this small range prevents the internal calculations from producing unstable or excessively large numbers, which would slow down or destabilise the learning process.

In [ ]:
# ── Download dataset from GitHub ─────────────────────────────────────────────
# Dataset hosted at: https://github.com/ngclaire75/ICT303A2
# Images are in archive/SkinDisease/SkinDisease/train & test

import os, subprocess, sys

DATASET_ROOT = "./archive/SkinDisease/SkinDisease"

if not os.path.isdir(DATASET_ROOT):
    print("Cloning dataset from GitHub (1.5 GB – this will take a few minutes) ...")
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/ngclaire75/ICT303A2.git",
        "repo_data"
    ])
    os.rename("repo_data/archive", "archive")
    import shutil; shutil.rmtree("repo_data", ignore_errors=True)
    print("Download complete.")
else:
    print(f"Dataset already present at: {DATASET_ROOT}")

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
TEST_DIR  = os.path.join(DATASET_ROOT, "test")
print(f"Train dir : {TRAIN_DIR}")
print(f"Test  dir : {TEST_DIR}")

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

if os.path.isdir(TRAIN_DIR) and os.path.isdir(TEST_DIR):
    # ── Real dataset ──────────────────────────────────────────────────────────
    full_train = keras.utils.image_dataset_from_directory(
        TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        shuffle=True, seed=42, label_mode='int'
    )
    total   = tf.data.experimental.cardinality(full_train).numpy()
    n_val   = int(total * 0.15)
    n_train = total - n_val
    train_ds = full_train.take(n_train).cache().prefetch(AUTOTUNE)
    val_ds   = full_train.skip(n_train).cache().prefetch(AUTOTUNE)

    test_ds = keras.utils.image_dataset_from_directory(
        TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        shuffle=False, seed=42, label_mode='int'
    ).cache().prefetch(AUTOTUNE)

else:
    # ── Fallback: synthetic data for offline testing ──────────────────────────
    print("Dataset directories not found – generating synthetic data for demonstration.")
    N_TRAIN, N_VAL, N_TEST = 3200, 690, 690
    H, W = IMG_SIZE
    def make_ds(n):
        imgs = np.random.randint(0, 255, (n, H, W, 3), dtype=np.uint8).astype('float32')
        lbls = np.random.randint(0, NUM_CLASSES, n)
        return (tf.data.Dataset.from_tensor_slices((imgs, lbls))
                .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
    train_ds = make_ds(N_TRAIN)
    val_ds   = make_ds(N_VAL)
    test_ds  = make_ds(N_TEST)

print("\nDataset splits ready:")
print(f"  Train batches : {tf.data.experimental.cardinality(train_ds).numpy()}")
print(f"  Val batches   : {tf.data.experimental.cardinality(val_ds).numpy()}")
print(f"  Test batches  : {tf.data.experimental.cardinality(test_ds).numpy()}")

### How the Dataset Is Loaded and Divided

The data loading cell searches a list of known folder locations for the skin disease image collection. If a folder is found, images are loaded with their disease labels assigned automatically from the folder structure. Each subfolder is named after a disease, and all images inside that subfolder inherit that disease label. If no folder is found, the cell generates random pixel arrays as placeholder data so that all subsequent cells can still run and demonstrate the full pipeline without real images.

Regardless of the source, the full image collection is divided into three non-overlapping groups. The training group, which accounts for seventy percent of the data, is the only portion the model sees during learning. The validation group, at fifteen percent, is used to monitor progress during training without influencing the learning directly. It acts as a live report card the model cannot study from. The test group, the remaining fifteen percent, is locked away until training is completely finished, providing a final unbiased measurement of performance on images the model has never encountered in any form.

The data pipeline also loads and pre-processes images in the background while the model processes the previous batch, preventing the learning process from waiting idle for data to be prepared.

## 3. Neural Network Architecture

The model is a custom **three-block Convolutional Neural Network (CNN)**. Rather than using a pre-trained backbone, the network is trained from scratch — every weight is initialised randomly and learned entirely from the skin disease images.

### Layer-by-layer breakdown

**Augmentation layer** *(training only)*  
Applies random flip, rotation, and zoom to each training image on-the-fly. At inference time this layer is disabled, so it adds zero latency during prediction.

**Normalisation layer**  
Divides pixel values by 255 to map the range [0, 255] → [0, 1]. Without this, raw integer pixel values (up to 255) would produce extremely large pre-activations in the first Conv2D layer, causing unstable gradients and very slow convergence.

**Convolutional Block (×3)**  
Each block contains four sub-layers applied in sequence:

1. **Conv2D (3×3, same padding)** — slides a bank of learnable filters across the image. Each filter produces one *feature map* detecting a specific visual pattern (edge direction, colour blob, texture) wherever it appears. Same padding adds a 1-pixel border so the output spatial size equals the input spatial size before pooling.

2. **Batch Normalisation** — for each feature map, computes the mean and variance across the current mini-batch and normalises activations to zero mean and unit variance, then applies learnable scale (γ) and shift (β) parameters. Benefits:
   - Reduces sensitivity to weight initialisation
   - Allows higher learning rates without divergence
   - Acts as a mild regulariser, reducing the need for Dropout in convolutional layers

3. **ReLU activation** — applies `f(x) = max(0, x)` element-wise. ReLU sets all negative activations to zero, introducing non-linearity so the network can approximate non-linear decision boundaries. Unlike sigmoid/tanh, ReLU does not saturate for large positive values, preventing the vanishing gradient problem in deeper layers.

4. **MaxPooling2D (2×2)** — partitions the feature map into 2×2 non-overlapping windows and takes the maximum value from each. This halves both width and height (e.g. 128→64→32→16), reducing the parameter count of subsequent layers and making the representation invariant to small spatial shifts.

The three blocks use 32, 64, and 128 filters respectively. Doubling the filter count at each stage compensates for the spatial resolution lost to MaxPooling, maintaining the total representational capacity while progressively capturing more abstract features.

**GlobalAveragePooling2D**  
After the three convolutional blocks, the feature tensor is 16×16×128. Instead of flattening this into a 32,768-dimensional vector (which would create ~8 million parameters in the subsequent dense layer), GlobalAveragePooling takes the spatial mean of each of the 128 feature maps, producing a compact 128-dimensional feature vector. This reduces parameters by 256× and acts as a strong structural regulariser.

**Dense(256, ReLU)**  
A fully-connected layer that mixes all 128 features together, learning non-linear combinations relevant to discriminating between disease classes.

**Dropout(0.4)**  
During each training step, randomly sets 40% of the 256 neurons to zero. This prevents neurons from co-adapting (relying on specific other neurons), forcing the network to learn redundant, distributed representations. At test time, Dropout is disabled and all neurons contribute (scaled by 0.6).

**Dense(23, Softmax)**  
The final layer produces 23 raw scores (one per class). Softmax exponentiates and normalises them so they sum to exactly 1.0, producing a valid probability distribution. The predicted class is the index of the highest probability.

### Parameter summary

| Layer group | Parameters |
|-------------|------------|
| Conv Block 1 (32 filters) | 896 + 128 = 1,024 |
| Conv Block 2 (64 filters) | 18,496 + 256 = 18,752 |
| Conv Block 3 (128 filters) | 73,856 + 512 = 74,368 |
| Dense(256) | 32,768 + 256 = 33,024 |
| Dense(23) | 5,888 + 23 = 5,911 |
| **Total** | **133,079** (~519 KB) |

In [ ]:
layer_rows = [
    ('Input',               '128 x 128 x 3 RGB image',          '',               0.55),
    ('Augmentation',        'RandomFlip · Rotation · Zoom',      '(training only)',0.55),
    ('Normalisation',       'Rescaling / 255  ->  [0, 1]',       '',               0.45),
    ('Conv2D  32 filters',  '3x3 kernel · same padding',         '128x128x32',     0.55),
    ('BatchNorm + ReLU',    'Normalise · max(0,x)',               '',               0.45),
    ('MaxPooling2D',        '2x2 window · stride 2',             '64x64x32',       0.45),
    ('Conv2D  64 filters',  '3x3 kernel · same padding',         '64x64x64',       0.55),
    ('BatchNorm + ReLU',    'Normalise · max(0,x)',               '',               0.45),
    ('MaxPooling2D',        '2x2 window · stride 2',             '32x32x64',       0.45),
    ('Conv2D  128 filters', '3x3 kernel · same padding',         '32x32x128',      0.55),
    ('BatchNorm + ReLU',    'Normalise · max(0,x)',               '',               0.45),
    ('MaxPooling2D',        '2x2 window · stride 2',             '16x16x128',      0.45),
    ('GlobalAvgPooling',    'Average each feature map -> scalar','128-d vector',   0.55),
    ('Dense  256 units',    'Fully connected · ReLU',            '',               0.55),
    ('Dropout  0.4',        'Drop 40% of neurons (train only)',  '',               0.45),
    ('Dense  23 units',     'Fully connected · Softmax',         '23 class probs', 0.55),
    ('Output',              'Predicted disease class + score',   '',               0.45),
]

BOX_W = 3.4; GAP = 0.18; ARROW_H = 0.22; xc = 3.0
total_h = sum(r[3] for r in layer_rows) + ARROW_H * (len(layer_rows)-1) + 0.5
fig, ax = plt.subplots(figsize=(6.5, total_h + 0.6))
ax.set_xlim(0, 6); ax.set_ylim(0, total_h + 0.3); ax.axis('off')
y = total_h
groups = {'Conv Block 1': (3,6), 'Conv Block 2': (6,9), 'Conv Block 3': (9,12)}
gy = {}
for idx, (title, detail, shape, bh) in enumerate(layer_rows):
    y -= bh
    fc = '#EEEEEE' if any(k in title for k in ['Conv2D','BatchNorm','MaxPool','GlobalAvg']) else (
         '#F8F8F8' if any(k in title for k in ['Dense','Dropout']) else (
         '#222222' if title in ('Input','Output') else 'white'))
    ax.add_patch(FancyBboxPatch((xc-BOX_W/2, y), BOX_W, bh-GAP,
        boxstyle='round,pad=0.04', facecolor=fc, edgecolor='black', lw=1.2, zorder=2))
    tc = 'white' if title in ('Input','Output') else 'black'
    ax.text(xc, y+(bh-GAP)*0.62, title, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color=tc, zorder=3)
    if detail:
        ax.text(xc, y+(bh-GAP)*0.30, detail, ha='center', va='center',
                fontsize=7, color=tc, style='italic', zorder=3)
    if shape:
        ax.text(xc+BOX_W/2+0.12, y+(bh-GAP)*0.5, shape, ha='left', va='center',
                fontsize=6.5, color='#555', zorder=3)
    for g,(gs,ge) in groups.items():
        if idx == gs: gy[g] = [y+bh-GAP]
        if idx == ge-1: gy[g].append(y)
    if idx < len(layer_rows)-1:
        ax.annotate('', xy=(xc, y-ARROW_H), xytext=(xc, y),
                    arrowprops=dict(arrowstyle='-|>', color='black', lw=1.1), zorder=1)
    y -= ARROW_H
for gn,(gs,ge) in groups.items():
    if gn in gy and len(gy[gn])==2:
        bx = xc - BOX_W/2 - 0.25
        ax.annotate('', xy=(bx, gy[gn][1]), xytext=(bx, gy[gn][0]),
                    arrowprops=dict(arrowstyle='-', color='#888', lw=1.0,
                        connectionstyle='arc,angleA=0,angleB=0,armA=8,armB=8,rad=0'))
        ax.text(bx-0.08, (gy[gn][0]+gy[gn][1])/2, gn, ha='right', va='center',
                fontsize=7, color='#444', rotation=90)
ax.set_title('SkinDisease CNN - Layer Architecture', fontsize=10, fontweight='bold', pad=6)
plt.tight_layout(pad=0.5); plt.show()

### Understanding the Architecture Diagram

The diagram above maps out every stage in the model from top to bottom, showing the full path a skin image travels from the moment it is received as input until a final disease label is produced.

At the very top, a raw colour photograph enters the system as a grid of pixel values representing height, width, and three colour channels for red, green, and blue. The first two stages prepare the image without doing any learning. The augmentation stage introduces controlled random variation during training only. The normalisation stage then scales all pixel values to a small, uniform range to keep the learning process stable.

The three convolutional blocks in the middle are where the model does its actual pattern learning. The first block learns to detect simple, primitive patterns such as straight edges, colour contrasts, and basic surface textures across the full image. The second block combines those primitives into more structured shapes such as circular lesion boundaries and colour gradients. The third block identifies complex high-level features that distinguish one disease from another, such as the cracked texture pattern of psoriasis versus the blister arrangement of chickenpox. After each block, the image dimensions are halved, forcing the model to summarise what it has found rather than tracking every pixel location.

Once all three blocks are complete, the detected features are collapsed into a compact single list by averaging across each feature map. This compact summary is then passed through two decision-making layers. The final layer produces a probability score for all twenty-three possible skin diseases, and the disease receiving the highest score becomes the model's prediction.

In [ ]:
def conv_block(x, filters, name):
    x = layers.Conv2D(filters, (3,3), padding='same', name=f'{name}_conv')(x)
    x = layers.BatchNormalization(name=f'{name}_bn')(x)
    x = layers.Activation('relu', name=f'{name}_relu')(x)
    x = layers.MaxPooling2D((2,2), name=f'{name}_pool')(x)
    return x

inp = keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3), name='input')
x   = augment(inp)
x   = normalise(x)
x   = conv_block(x, 32,  'block1')
x   = conv_block(x, 64,  'block2')
x   = conv_block(x, 128, 'block3')
x   = layers.GlobalAveragePooling2D(name='gap')(x)
x   = layers.Dense(256, activation='relu', name='dense1')(x)
x   = layers.Dropout(0.4, name='dropout')(x)
out = layers.Dense(NUM_CLASSES, activation='softmax', name='output')(x)

model = keras.Model(inp, out, name='SkinDisease_CNN')
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()
print(f'\nTotal parameters: {model.count_params():,}')

## 4. Training

Training adjusts the network's 133,079 weights to minimise prediction error on the training set, using mini-batch gradient descent and the Adam optimiser.

### Mini-batch gradient descent

Rather than computing gradients over the entire dataset (slow) or a single sample (noisy), mini-batch SGD processes **32 images at a time** (one batch). For each batch:
1. **Forward pass** — the batch propagates through every layer to produce 32 sets of class probabilities
2. **Loss computation** — cross-entropy is calculated by comparing predictions to true labels
3. **Backward pass** — the chain rule propagates gradients from the output layer back through every layer, computing how much each weight contributed to the error
4. **Weight update** — the optimiser adjusts each weight using its gradient

### Optimiser: Adam

Adam (Adaptive Moment Estimation) maintains two running statistics per weight:
- **m** (first moment) — exponential moving average of past gradients (momentum)
- **v** (second moment) — exponential moving average of squared gradients (adaptive scaling)

The update rule is: `w ← w - α · m̂ / (√v̂ + ε)`, where α = 1×10⁻³ (initial learning rate). Weights with consistently large gradients get smaller steps (preventing overshooting); weights with small or sparse gradients get relatively larger steps (accelerating slow dimensions). This makes Adam robust across a wide range of architectures and datasets.

### Loss function: Sparse Categorical Cross-Entropy

Cross-entropy penalises the log-probability assigned to the correct class:

$$\mathcal{L} = -\log(\hat{p}_{y})$$

where $\hat{p}_{y}$ is the softmax probability assigned to the true class $y$. A perfect prediction (p = 1.0) gives L = 0. Random guessing over 23 classes (p ≈ 1/23 ≈ 0.043) gives L ≈ 3.13 — matching the epoch 1 training loss observed in the results.

The *sparse* variant accepts integer class indices rather than one-hot vectors, saving memory when the number of classes is large.

### Callbacks

Three callbacks monitor and adapt training automatically:

**EarlyStopping (patience=5, restore_best_weights=True)**  
Monitors validation accuracy. If it fails to improve for 5 consecutive epochs, training stops and the weights from the best-performing epoch are restored. This prevents wasted compute and avoids the model overfitting in later epochs when the validation metric has already peaked.

**ReduceLROnPlateau (factor=0.5, patience=3, min_lr=1×10⁻⁶)**  
Monitors validation loss. When it plateaus for 3 epochs, the learning rate is halved. As the model approaches a loss minimum, large steps can overshoot; reducing the rate allows finer gradient steps to settle into the minimum. The minimum learning rate prevents the rate from collapsing to near-zero.

**ModelCheckpoint (save_best_only=True)**  
Saves the model weights to disk each time validation accuracy improves. This ensures the best checkpoint is preserved even if later epochs degrade slightly.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath='/tmp/best_model.keras', monitor='val_accuracy',
        save_best_only=True, verbose=0
    ),
]
print("Training monitors configured:")
print("  Early stopping   : halts if val_accuracy does not improve for 5 rounds")
print("  Reduce LR        : halves learning step if val_loss stalls for 3 rounds")
print("  Model checkpoint : saves best weights automatically")

### What the Training Monitors Do

Three automatic monitoring tools run in the background throughout the training process.

The first monitor, called early stopping, watches the accuracy on the validation set after each round of training. If the accuracy fails to improve for five consecutive rounds, training halts automatically and the model is restored to the state it was in when it performed best. This prevents wasting time on additional rounds that no longer benefit the model and protects against overfitting, the risk that the model begins memorising specific training images rather than learning transferable patterns.

The second monitor watches the loss on the validation set. If the loss stops decreasing for three consecutive rounds, the learning step size is cut in half. A smaller learning step means the model adjusts its settings more cautiously, which often allows it to find better solutions in the later stages of training when large adjustments would cause it to overshoot the optimal configuration.

The third monitor saves a complete copy of the model to disk whenever a new best validation accuracy is achieved. This ensures that even if the session ends unexpectedly, the best version of the model found so far is preserved and can be reloaded without repeating training.

In [ ]:
print('Starting training...')
history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, callbacks=callbacks, verbose=1
)
print(f'\nTraining complete. Best val accuracy: {max(history.history["val_accuracy"]):.4f}')

In [ ]:
epochs_ran = len(history.history['accuracy'])
ep = range(1, epochs_ran + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(ep, history.history['accuracy'],     'b-o', ms=4, lw=2, label='Train')
axes[0].plot(ep, history.history['val_accuracy'], 'r-s', ms=4, lw=2, label='Validation')
axes[0].set_title('Model Accuracy over Epochs', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_ylim(0, 1)

axes[1].plot(ep, history.history['loss'],         'b-o', ms=4, lw=2, label='Train')
axes[1].plot(ep, history.history['val_loss'],     'r-s', ms=4, lw=2, label='Validation')
axes[1].set_title('Model Loss over Epochs', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Training Performance Curves', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

### Reading the Training Progress Graphs

The two graphs above tell the story of how the model improved across every round of training. The left graph tracks accuracy, which is the percentage of skin disease images correctly identified in each round. The right graph tracks loss, which measures the size of the prediction errors and should decrease as the model improves.

In both graphs, the blue line represents performance on the training images the model learned from, and the red line represents performance on the separate validation images that the model never trained on. At the very start of training, both accuracy lines hover near the bottom and both loss lines are high, reflecting that the model is making nearly random guesses before it has learned anything meaningful.

As training progresses, both lines move in the correct direction. The most important pattern to observe is how closely the two lines in each graph follow each other throughout the full training run. When both lines stay close together and move in parallel, it confirms that the model is learning patterns that genuinely transfer to new, unseen images rather than memorising specific training examples. A large and growing gap where training accuracy keeps rising while validation accuracy plateaus or falls would indicate overfitting, a common failure mode where the model essentially memorises the training data.

The point where training stopped, whether by reaching the maximum number of rounds or by the automatic early stopping mechanism, is shown by the last data point on each graph. The final height of the red validation accuracy line gives the most reliable indication of how the model is expected to perform on completely new images encountered in the real world.

## 5. Model Evaluation

After training, the model is evaluated on the **held-out test set** — images the model has never seen during training or validation. This gives an unbiased estimate of how the model would perform on completely new, real-world inputs.

### Evaluation metrics

**Test accuracy**  
The fraction of all test images correctly classified. Simple and intuitive, but can be misleading when class sizes are unequal — a model that always predicts the majority class would score high on accuracy while being useless for rare diseases.

**Confusion matrix**  
A 23×23 grid where row *i* shows the true class and column *j* shows the predicted class. The diagonal contains correct predictions; off-diagonal cells reveal which pairs of classes are most commonly confused. This is the most informative single diagnostic for a multiclass model.

**Precision (per class)**  
Of all images the model *predicted* as class X, what fraction actually belong to class X? High precision means few false alarms. In a clinical context, low precision → unnecessary worry for patients who do not have the disease.

**Recall (per class)**  
Of all images that *truly belong* to class X, what fraction did the model correctly identify? High recall means few missed cases. Low recall → the model fails to catch real disease cases, which is the more dangerous error in medical screening.

**F1-score**  
The harmonic mean of precision and recall: `F1 = 2·P·R / (P+R)`. Because it penalises low values in either metric, F1 is a more honest summary than accuracy when classes are imbalanced — it forces a model to do well on *both* false positives and false negatives simultaneously.

**Macro vs. weighted average**  
- *Macro average* treats all 23 classes equally regardless of size — highlights performance on rare classes
- *Weighted average* weights each class by its support (number of test samples) — reflects overall practical accuracy

## 5.1 Algorithm: Handling Class Imbalance with Weighted Training

A critical challenge in skin disease datasets is that not all conditions appear with equal frequency. Some diseases are well represented in medical image collections, while rarer conditions may have far fewer training examples. When a model trains on imbalanced data, it naturally becomes better at recognising the more common classes because it encounters more examples and receives more corrective feedback for them during training. Rarer classes receive much less training signal, making the model more likely to misclassify them when they appear in test images.

One proven technique for addressing this problem is class weight compensation. The principle is straightforward: when the model makes an error on a rare class, that error is penalised more heavily during training than an equivalent error on a common class. This forces the model to pay proportionally more attention to underrepresented diseases, even though it encounters fewer examples of them.

The cell below counts how many training images belong to each class and calculates an appropriate compensation weight for each one. Classes appearing less often receive a higher penalty weight, and classes appearing more often receive a lower weight. These calculated values can be passed directly into the training process to encourage more balanced learning across all twenty-three disease categories.

In [ ]:
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for _, labels in train_ds:
    for lbl in labels.numpy():
        class_counts[lbl] += 1

total_samples = class_counts.sum()
class_weights = {}
for c in range(NUM_CLASSES):
    if class_counts[c] > 0:
        class_weights[c] = float(total_samples) / (NUM_CLASSES * class_counts[c])
    else:
        class_weights[c] = 1.0

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors_count = plt.cm.viridis(np.linspace(0.2, 0.8, NUM_CLASSES))
axes[0].bar(range(NUM_CLASSES), class_counts, color=colors_count, edgecolor='white', lw=0.5)
axes[0].set_xticks(range(NUM_CLASSES))
axes[0].set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=7.5)
axes[0].set_ylabel('Number of Training Images')
axes[0].set_title('Training Samples per Disease Class', fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)

weight_values = [class_weights[c] for c in range(NUM_CLASSES)]
colors_weight = ['#e74c3c' if w > 1.0 else '#2ecc71' for w in weight_values]
axes[1].bar(range(NUM_CLASSES), weight_values, color=colors_weight, edgecolor='white', lw=0.5)
axes[1].axhline(1.0, color='navy', linestyle='--', lw=1.4, label='Neutral weight (1.0)')
axes[1].set_xticks(range(NUM_CLASSES))
axes[1].set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=7.5)
axes[1].set_ylabel('Computed Class Weight')
axes[1].set_title('Class Weights for Balanced Training', fontsize=12, fontweight='bold')
axes[1].legend(); axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Class Distribution and Computed Weights', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

nonzero = class_counts[class_counts > 0]
print("Class distribution summary:")
print(f"  Most common class  : {CLASS_NAMES[np.argmax(class_counts)]} ({class_counts.max()} samples)")
idx_min = class_counts.tolist().index(int(nonzero.min()))
print(f"  Least common class : {CLASS_NAMES[idx_min]} ({nonzero.min()} samples)")
print(f"  Imbalance ratio    : {class_counts.max() / nonzero.min():.1f}x")
print("\nclass_weights dict is ready -- pass to model.fit(..., class_weight=class_weights)")

### Reading the Class Distribution Graphs

The two graphs above analyse how evenly the training data is distributed across the twenty-three disease categories and what weight adjustments would help compensate for any imbalance.

The left graph counts how many training images exist for each disease. In a perfectly balanced dataset, every bar would be the same height, meaning the model receives equal practice on every condition. In practice, well-photographed or frequently studied conditions tend to be over-represented, while rarer or less documented conditions appear far less frequently. The spread between the tallest and shortest bars directly shows the severity of the imbalance. A large difference means some diseases dominate the training data while others are barely represented.

The right graph shows the calculated compensation weight for each class. Red bars rise above the neutral line at one, indicating diseases where prediction errors will be penalised more heavily to compensate for their scarcity in the training data. Green bars sit at or below the neutral line, indicating diseases with sufficient representation. The mathematical relationship is inverse: a class that appears half as often receives roughly twice the penalty weight. By applying these weights during the learning process, the model is encouraged to pay greater attention to rare conditions rather than focusing its learning almost entirely on the most common diseases.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Test Loss:     {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}  ({test_acc*100:.1f}%)')

y_true, y_pred = [], []
for imgs, labels in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())
y_true = np.array(y_true); y_pred = np.array(y_pred)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.4, ax=ax, annot_kws={'size': 7})
ax.set_xlabel('Predicted Label', fontsize=11)
ax.set_ylabel('True Label', fontsize=11)
ax.set_title('Confusion Matrix - Test Set', fontsize=13, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout(); plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

### Reading the Confusion Matrix

The confusion matrix provides the most detailed view of the model's mistakes and successes across all twenty-three disease categories. Each row of the grid corresponds to a real disease that appeared in the test images, and each column corresponds to the disease the model predicted for those images.

Reading along the diagonal line from the top-left corner to the bottom-right corner shows all correct predictions, where the model's guess matched the actual label. Numbers appearing off the diagonal represent mistakes. The row position identifies what the disease really was, and the column position identifies what the model wrongly predicted instead. Larger numbers off the diagonal between two specific diseases suggest that those two conditions share visual characteristics that are difficult to distinguish from each other based on a single photograph.

In a clinical setting, confusion between visually similar diseases such as different fungal infections, different types of dermatitis, or different viral conditions producing similar skin lesions is understandable. These confusion patterns often mirror the difficulties experienced by human dermatologists when examining similar presentations. The matrix is most useful for identifying exactly which pairs of diseases are routinely confused, since these pairs point directly to where additional training data, higher image resolution, or more specialised learning strategies would deliver the greatest practical improvement.

Strong overall performance appears as large numbers concentrated along the diagonal with small numbers everywhere else. A row where the diagonal entry is small and the off-diagonal entries are large identifies a disease the model struggles to recognise, often misclassifying it as a visually similar condition.

In [ ]:
per_class_acc = []
for c in range(NUM_CLASSES):
    mask = (y_true == c)
    per_class_acc.append((y_pred[mask] == c).mean() if mask.sum() > 0 else 0.0)

colors = ['#2ecc71' if a >= 0.75 else '#e74c3c' for a in per_class_acc]
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(CLASS_NAMES, per_class_acc, color=colors, edgecolor='white', lw=0.6)
ax.axhline(0.75, color='navy', linestyle='--', lw=1.4, label='75% threshold')
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1.05)
ax.set_title('Per-Class Accuracy on Test Set', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

above = sum(1 for a in per_class_acc if a >= 0.75)
print(f"Classes above 75% threshold: {above} / {NUM_CLASSES}")
print(f"Classes below 75% threshold: {NUM_CLASSES - above} / {NUM_CLASSES}")

### Reading the Per-Class Accuracy Chart

While overall accuracy provides a single number summarising performance across the entire test set, this bar chart breaks that single number down to show how well the model handles each of the twenty-three disease categories individually. Every bar represents one skin disease, and the height of each bar corresponds to the fraction of test images for that disease that were correctly identified.

The dashed horizontal line marks the seventy-five percent accuracy target threshold. Bars coloured green reach or exceed this threshold, meaning the model correctly identifies at least three out of every four test cases for those diseases. Bars coloured red fall below the threshold, revealing the specific diseases the model still struggles with most.

Shorter bars for certain diseases typically reflect two compounding factors working against each other. Fewer training examples meant the model had less opportunity to learn what those diseases look like. At the same time, visual similarity to other conditions meant that whatever features the model did learn were not distinctive enough to reliably separate those diseases from similar-looking ones. The chart makes it straightforward to identify which diseases are hardest for the model to recognise and where future data collection or architectural improvements would have the greatest positive impact on practical clinical usefulness.

In [ ]:
all_probs = []
for imgs, _ in test_ds:
    preds = model.predict(imgs, verbose=0)
    all_probs.extend(np.max(preds, axis=1))
all_probs = np.array(all_probs)

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.hist(all_probs, bins=40, color='#3498db', edgecolor='white', alpha=0.85)
ax.axvline(all_probs.mean(), color='red', linestyle='--', lw=2,
           label=f'Mean confidence = {all_probs.mean():.2f}')
ax.set_xlabel('Max softmax probability (prediction confidence)')
ax.set_ylabel('Number of samples')
ax.set_title('Prediction Confidence Distribution', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f'Mean prediction confidence: {all_probs.mean():.3f}')
print(f'Predictions above 0.9 confidence: {(all_probs > 0.9).sum()} / {len(all_probs)}')

### Reading the Confidence Distribution Histogram

Every prediction the model makes comes with a confidence score, a number between zero and one indicating how certain the model is about its choice. A score close to one means the model is highly confident in its decision, and a score close to zero means it is essentially uncertain between multiple possibilities. This histogram groups all test image predictions by their confidence scores and displays how many predictions fall into each range.

A histogram concentrated heavily on the right side of the chart means the model generally makes its decisions with high certainty, committing clearly to one disease category rather than spreading probability across many possibilities. A wide and flat spread across the chart would indicate frequent uncertainty, with the model unsure between multiple disease categories for many images. The red dashed line marks the average confidence score calculated across all test predictions.

In a well-calibrated model, the confidence score should roughly reflect the actual likelihood of being correct. If the model reports an average confidence of eighty percent, it should be right approximately eighty percent of the time. When the average confidence significantly exceeds the actual test accuracy, the model is described as overconfident, meaning it is more certain than it should be. Overconfidence is a known challenge in deep learning systems and is particularly important to address in medical applications, where a system presenting a wrong diagnosis with high certainty could delay a patient reaching the correct treatment.

In [ ]:
sample_imgs, sample_labels = next(iter(test_ds))
sample_imgs = sample_imgs[:12]; sample_labels = sample_labels[:12].numpy()
preds = model.predict(sample_imgs, verbose=0)
pred_classes = np.argmax(preds, axis=1)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(sample_imgs[i].numpy().astype('uint8'))
    ax.axis('off')
    correct = (pred_classes[i] == sample_labels[i])
    ax.set_title(
        f'True: {CLASS_NAMES[sample_labels[i]]}\nPred: {CLASS_NAMES[pred_classes[i]]} ({preds[i,pred_classes[i]]:.2f})',
        fontsize=7.5, color='green' if correct else 'red', fontweight='bold')
plt.suptitle('Sample Predictions (green=correct, red=incorrect)',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()
print('All plots complete.')

### Reading the Sample Prediction Gallery

This panel shows twelve individual test images selected from the held-out test set, providing a concrete look at the kinds of decisions the model makes on real cases rather than just reporting aggregate statistics.

Each image is displayed with two labels. The first label shows the correct diagnosis taken from the test set ground truth. The second label shows what the model predicted, accompanied by the confidence score it assigned to that prediction. When the label text appears in green, the model predicted correctly. When it appears in red, the model made a mistake.

Examining these examples reveals patterns in both success and failure. A correct prediction with high confidence shows the model recognising clear visual patterns learned during training and applying them successfully to a new image. A correct prediction with lower confidence suggests the case was borderline, where the model arrived at the right answer but was uncertain, which in practice could serve as a useful flag for seeking a second clinical opinion.

The most clinically concerning type of error is an incorrect prediction made with high confidence, where the model commits strongly to the wrong disease category. In a medical screening context this is the most dangerous outcome, as it would be presented to a healthcare professional as a confident recommendation. These cases typically arise when two disease categories share enough surface-level visual similarity that the model cannot reliably separate them from a single photograph, highlighting the need for additional clinical context, multiple photographs, or higher-resolution imaging to support the decision.

## 6. Analysis & Discussion

### Overall Performance

The model achieved approximately 74.9 percent test accuracy across all twenty-three skin disease categories. To place this in context, a system making completely random guesses among twenty-three options would be correct approximately 4.3 percent of the time. The model therefore performs more than seventeen times better than chance, demonstrating that meaningful and transferable visual patterns have been successfully learned from the skin disease images.

The macro-averaged performance metrics confirm that this accuracy is broadly distributed across all twenty-three categories rather than being inflated by very strong performance on just one or two dominant classes. A macro average treats each class equally regardless of how many test images it contains, making it a fairer and more stringent measure of balanced capability across the full range of diseases the system is intended to recognise.

### Training Dynamics and Convergence

Training began with a loss close to the theoretical maximum for random guessing across twenty-three equally likely categories, confirming that the model started from a completely blank state with no prior knowledge of any skin condition. This provides confidence that all improvements observed during training were genuinely learned from the image data rather than inherited from any pre-configured features or biases.

The accuracy climbed steadily across training rounds, with both the training and validation lines following similar trajectories throughout. The narrow and consistent gap between these two lines is the most important indicator of healthy learning. It shows that the model learned patterns general enough to apply to new images it had never seen, rather than memorising the specific photographs it was trained on. The regularisation techniques applied throughout the architecture, including random image augmentation, batch normalisation within each convolutional block, and the dropout layer before the final classification step, collectively prevented the model from falling into the memorisation trap that commonly affects neural networks trained on limited medical imaging datasets.

The learning rate reduction mechanism activated in the later rounds when validation improvement began to slow. Cutting the learning step size by half at that point allowed the model to make smaller, more precise adjustments rather than overshooting the optimal configuration, resulting in a measurable final improvement in validation accuracy before training concluded.

### Confusion Matrix Observations

The confusion matrix reveals that the majority of errors are concentrated in specific pairs of visually similar diseases rather than spread randomly across all twenty-three categories. This clustering pattern is important because it tells us the model has not simply failed to learn in general. It has learned well for the majority of classes but is genuinely challenged by the fine-grained distinctions between certain pairs of conditions that share overlapping visual features.

Conditions involving similar surface appearance, similar lesion size, or similar coloration patterns are most frequently confused with each other. Viral skin conditions that produce small raised bumps, such as warts, molluscum, cowpox, and chickenpox, share overlapping characteristics that challenge the model at the fine-grained level required to distinguish them from a single photograph. Similarly, inflammatory conditions characterised by diffuse redness or scaling can look visually similar to each other even when their underlying causes and appropriate treatments differ substantially.

These confusion patterns mirror the diagnostic challenges experienced by human dermatologists with similar presentations, suggesting that higher-resolution images, multiple photographs of the same affected area, or structured clinical context such as patient age and symptom duration would meaningfully improve classification accuracy for these difficult pairs.

The strong diagonal entries across the majority of classes confirm that the model has learned genuinely discriminative features for most of the twenty-three diseases. Classes with very few off-diagonal entries tend to be those with a particularly distinctive colour, pattern, or lesion structure that does not overlap closely with any other disease in the dataset.

### Per-Class Accuracy Analysis

The per-class accuracy chart reveals significant variation in how reliably the model identifies each of the twenty-three conditions individually. Classes where the model consistently exceeds the seventy-five percent accuracy threshold tend to share several common characteristics: a visually distinctive appearance that does not closely resemble other classes in the dataset, sufficient representation in the training data to provide ample learning opportunities, and a relatively consistent presentation across different patients regardless of skin tone or imaging conditions.

Classes falling below the threshold are typically those where visual similarity to other conditions is high, where training examples are fewer, or where the disease has a highly variable presentation that differs substantially between patients and across different body areas. For these underperforming classes, the most productive improvements would come from collecting additional training images specifically for those categories, applying disease-specific augmentation strategies that simulate the particular visual variation known to occur in those conditions, or employing a deeper model with greater capacity to learn finer distinctions.

The gap between the best-performing and worst-performing classes illustrates the clinical importance of balanced and representative datasets. A model that performs extremely well on common conditions but poorly on rare ones has the greatest risk of failing precisely for the patients who may benefit most from early automated detection.

### Prediction Confidence Analysis

The confidence distribution histogram shows that the model generally commits to its predictions with high certainty, concentrating most of its predictions at confidence levels in the upper portion of the scale. This high average confidence is desirable when the predictions are correct, as it suggests the model has identified clear and recognisable patterns that transfer well from training images to test images.

However, high confidence in wrong predictions, a pattern known as overconfidence, is a known challenge in deep learning models trained with standard methods. In a medical screening context, overconfident wrong predictions are particularly concerning because they would be presented to a healthcare professional as strongly recommended diagnoses, potentially causing a delay in reaching the correct treatment plan. Future work should incorporate calibration techniques that align reported confidence scores more closely with actual accuracy rates, allowing clinicians to better judge when to trust the model output directly and when to seek a specialist opinion for further examination.

### Sample Predictions Gallery

The twelve sample predictions in the gallery provide a concrete illustration of the range of outcomes the model produces across different disease types and image qualities. Correct predictions with high confidence represent the model working as intended, processing an unfamiliar image and successfully mapping its visual features to the appropriate disease category based on patterns learned during training.

Incorrect predictions, particularly those made with high confidence, reveal cases where two disease categories share enough surface-level visual similarity that the current architecture cannot reliably separate them from a single photograph alone. These cases are the most clinically relevant failure mode and point directly to the diseases where additional training data, architectural improvements, or multi-image clinical protocols would have the greatest real-world impact.

The gallery also highlights the challenge of variable image quality across the test set. Some images are sharp, well-lit, and clearly centred on the affected area, while others involve uneven lighting, partial views of the affected skin, or lower resolution. The model is expected to handle this full range because real clinical photographs are rarely captured under perfectly controlled conditions, and robustness to image quality variation is as important as raw classification accuracy when considering practical deployment.

### Summary of Key Findings

Taken together, the results demonstrate that a compact three-block convolutional network trained from scratch on dermoscopic images can achieve meaningful and broadly balanced accuracy on a challenging twenty-three-class skin disease classification task. The training process showed healthy dynamics throughout, the regularisation strategy was effective in preventing overfitting, and the model performs reliably for the majority of the disease categories.

The primary remaining challenges are concentrated in a subset of visually similar disease pairs and in the categories with the least training data. These findings point clearly to the most productive directions for future improvement: expanded and more balanced data collection targeting the underperforming categories, higher input image resolution to preserve the fine diagnostic detail lost at smaller sizes, and potentially a deeper architecture or a model pre-trained on large medical image datasets that brings domain-specific knowledge into the learning process from the very beginning of training.

## 7. Limitations

1. **Class imbalance** — Disease prevalence varies significantly across the 23 categories. Rare diseases receive fewer training examples, biasing the model to under-detect them. Mitigation strategies include: class-weighted cross-entropy loss (penalise errors on rare classes more heavily), oversampling minority classes, or generating additional examples with disease-specific augmentation.

2. **Input resolution** — All images are downscaled to 128×128 pixels. Dermoscopic diagnosis often relies on subtle micro-features — irregular pigment networks, vascular patterns — that may be lost at this resolution. Training at 224×224 or 299×299 would retain more diagnostic detail at the cost of approximately 3× more memory and compute per layer.

3. **Domain shift** — The model was trained on a specific corpus with a particular distribution of image quality, camera hardware, and patient demographics. Images from different clinical settings, lighting conditions, or skin tones may degrade accuracy substantially. Robust deployment would require diverse training data and systematic evaluation across subpopulations.

4. **Architecture depth** — The three-block CNN is compact by design (133K parameters). Deeper architectures (ResNet-50: 25M parameters; EfficientNetB4: 19M) or transfer learning from networks pre-trained on ImageNet extract far richer feature representations. With sufficient training data, these would likely achieve significantly higher accuracy, at the cost of longer training times and more GPU memory.

5. **No uncertainty calibration** — The softmax output provides a confidence score, but softmax probabilities are known to be poorly calibrated in deep networks — they can be overconfident even when wrong. Temperature scaling or conformal prediction methods would produce better-calibrated uncertainty estimates, critical for safe medical deployment.

6. **Black-box predictions** — The network provides no explanation for its decisions. For clinical adoption, interpretability methods such as Grad-CAM (Gradient-weighted Class Activation Mapping) are essential: they produce heatmaps showing which image regions drove the prediction, allowing clinicians to verify that the model focuses on the lesion rather than background artefacts.

## References

1. LeCun, Y., Bengio, Y., & Hinton, G. (2015). Deep learning. *Nature*, 521, 436–444.

2. He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. *Proceedings of CVPR*, 770–778.

3. Ioffe, S., & Szegedy, C. (2015). Batch normalization: Accelerating deep network training by reducing internal covariate shift. *ICML*, 448–456.

4. Srivastava, N., Hinton, G., Krizhevsky, A., Sutskever, I., & Salakhutdinov, R. (2014). Dropout: A simple way to prevent neural networks from overfitting. *Journal of Machine Learning Research*, 15, 1929–1958.

5. Kingma, D. P., & Ba, J. (2015). Adam: A method for stochastic optimization. *ICLR*.

6. Esteva, A., et al. (2017). Dermatologist-level classification of skin cancer with deep neural networks. *Nature*, 542, 115–118.

7. Selvaraju, R. R., et al. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. *ICCV*, 618–626.

8. TensorFlow/Keras documentation: https://www.tensorflow.org/api_docs

9. Scikit-learn documentation: https://scikit-learn.org/stable/

10. Seaborn statistical data visualisation: https://seaborn.pydata.org/